<a href="https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ML-07 Setup
# Connect to the FlyRank warehouse through DuckDB.
# HF_TOKEN must be stored securely in Colab Secrets.

%pip -q install duckdb huggingface_hub

import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Add your READ Hugging Face token "
        "to Colab Secrets as HF_TOKEN."
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

PERFORMANCE = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet'"
    ")"
)

print("Warehouse connection ready.")
print("Development window: March 2026")

Warehouse connection ready.
Development window: March 2026


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


### Rule

I will prioritize content that has meaningful search visibility but shows weaker search position and/or lower click response. The rule is intended as a transparent review-prioritization baseline, not a prediction of future performance.

The score will combine two observable signals from the March 2026 development window:

* **Search impressions** — higher impressions indicate that the content is receiving measurable search exposure.
* **Average search position** — a worse average position indicates that the content may have room for search improvement.

### Reason codes

* `visible_but_weak_position` — the content has meaningful impressions and a relatively weak average search position.
* `visible_and_clickable` — the content has meaningful impressions and clicks and is suitable for review.
* `review_other` — the content does not meet either stronger condition.

The action labels will be:

* `PRIORITIZE_REVIEW`
* `REVIEW`
* `MONITOR`

The score is deliberately hand-written rather than learned from labels or future outcomes.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-07 Section 1
# Signal checks using the March 2026 warehouse partition.

# Signal 1: Search impressions
impression_check = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_impressions = 0 THEN '0'
            WHEN gsc_impressions < 100 THEN '1-99'
            WHEN gsc_impressions < 500 THEN '100-499'
            WHEN gsc_impressions < 1000 THEN '500-999'
            ELSE '1000+'
        END AS impressions_bucket,
        COUNT(*) AS n
    FROM {PERFORMANCE}
    WHERE gsc_impressions IS NOT NULL
    GROUP BY 1
    ORDER BY
        CASE impressions_bucket
            WHEN '0' THEN 1
            WHEN '1-99' THEN 2
            WHEN '100-499' THEN 3
            WHEN '500-999' THEN 4
            WHEN '1000+' THEN 5
        END
""").df()

print("Signal 1: Search impressions")
display(impression_check)


# Signal 2: Average search position
position_check = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position = 0 THEN 'no_data'
            WHEN gsc_avg_position <= 10 THEN '1-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            WHEN gsc_avg_position <= 50 THEN '21-50'
            ELSE '50+'
        END AS position_bucket,
        COUNT(*) AS n
    FROM {PERFORMANCE}
    WHERE gsc_avg_position IS NOT NULL
    GROUP BY 1
    ORDER BY
        CASE position_bucket
            WHEN 'no_data' THEN 1
            WHEN '1-10' THEN 2
            WHEN '11-20' THEN 3
            WHEN '21-50' THEN 4
            WHEN '50+' THEN 5
        END
""").df()

print("Signal 2: Average search position")
display(position_check)

print("\nVerdict — Impressions: CONFIRMED")
print("Verdict — Average search position: CONFIRMED")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1: Search impressions


,impressions_bucket,n
0,0,6230317
1,1-99,2972453
2,100-499,537157
3,500-999,69032
4,1000+,32419


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 2: Average search position


,position_bucket,n
0,no_data,163189
1,1-10,2020295
2,11-20,519223
3,21-50,631491
4,50+,276863



Verdict — Impressions: CONFIRMED
Verdict — Average search position: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


I will aggregate the March 2026 daily performance data to one row per client and content item. The baseline score uses only signals available in the March development window.

The score gives higher priority to content with meaningful search visibility and weaker average search position. Each row receives one reason code and one action label.

No future-window information, label-derived fields, or product-decision flags are used.

The resulting ranked queue will be written to:

`work/outputs/baseline_action_score.csv`


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-07 Section 2
# Build the transparent baseline score and ranked queue.

import os

# Aggregate March daily data to client × content level.
baseline_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position

    FROM {PERFORMANCE}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print(f"Rows in baseline frame: {len(baseline_df):,}")


# ---------------------------------------------------------
# Transparent hand-written rule
# ---------------------------------------------------------

# Signal 1: meaningful search visibility
baseline_df["visible"] = (
    baseline_df["impressions"] >= 500
).astype(int)

# Signal 2: weaker search position
baseline_df["weak_position"] = (
    baseline_df["avg_position"] > 20
).fillna(False).astype(int)

# Supporting click signal
baseline_df["has_clicks"] = (
    baseline_df["clicks"] > 0
).fillna(False).astype(int)


# Score:
# - visible + weak position = highest priority
# - visible + clicks = secondary review
# - everything else = monitor

baseline_df["score"] = (
    baseline_df["visible"] * 2
    + baseline_df["weak_position"] * 2
    + baseline_df["has_clicks"]
)


# ---------------------------------------------------------
# One reason code
# ---------------------------------------------------------

baseline_df["reason_code"] = np.select(
    [
        (baseline_df["visible"] == 1) &
        (baseline_df["weak_position"] == 1),

        (baseline_df["visible"] == 1) &
        (baseline_df["has_clicks"] == 1)
    ],
    [
        "visible_but_weak_position",
        "visible_and_clickable"
    ],
    default="review_other"
)


# ---------------------------------------------------------
# Action label
# ---------------------------------------------------------

baseline_df["action"] = np.select(
    [
        baseline_df["score"] >= 4,
        baseline_df["score"] >= 2
    ],
    [
        "PRIORITIZE_REVIEW",
        "REVIEW"
    ],
    default="MONITOR"
)


# Rank highest score first.
# Use impressions as a secondary ordering signal.
baseline_df = baseline_df.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_df["rank"] = np.arange(1, len(baseline_df) + 1)


# Keep the final queue fields.
queue = baseline_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "impressions",
        "clicks",
        "avg_position"
    ]
].copy()


# ---------------------------------------------------------
# Write output
# ---------------------------------------------------------

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

queue.to_csv(output_path, index=False)

print(f"Queue written to: {output_path}")
print(f"Total ranked rows: {len(queue):,}")

display(queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in baseline frame: 331,437
Queue written to: work/outputs/baseline_action_score.csv
Total ranked rows: 331,437


,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions,clicks,avg_position
0,1,client_23a62021009f63c4,content_36e53e9c707674fc,5,visible_but_weak_position,PRIORITIZE_REVIEW,194579.0,242.0,32.766674
1,2,client_20259bd6705d81d4,content_82e35c4845e6c391,5,visible_but_weak_position,PRIORITIZE_REVIEW,143907.0,60.0,22.558608
2,3,client_23a62021009f63c4,content_3df3f32f3fd58dea,5,visible_but_weak_position,PRIORITIZE_REVIEW,140156.0,197.0,23.335465
3,4,client_23a62021009f63c4,content_df47d1b976106de4,5,visible_but_weak_position,PRIORITIZE_REVIEW,131707.0,163.0,24.355625
4,5,client_23a62021009f63c4,content_bdf60c86117079be,5,visible_but_weak_position,PRIORITIZE_REVIEW,112429.0,12.0,30.769353
5,6,client_23a62021009f63c4,content_661a7734f691bef5,5,visible_but_weak_position,PRIORITIZE_REVIEW,110424.0,73.0,23.888656
6,7,client_23a62021009f63c4,content_cae701a83cad5e36,5,visible_but_weak_position,PRIORITIZE_REVIEW,98572.0,242.0,23.730705
7,8,client_23a62021009f63c4,content_559cdd76da9306de,5,visible_but_weak_position,PRIORITIZE_REVIEW,97378.0,2.0,36.712074
8,9,client_20259bd6705d81d4,content_9fff53e827550f9d,5,visible_but_weak_position,PRIORITIZE_REVIEW,94673.0,475.0,22.469914
9,10,client_fef1a8f436438636,content_ba462518dad435fc,5,visible_but_weak_position,PRIORITIZE_REVIEW,91391.0,46.0,27.355006


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



I will inspect the first 20 rows of the ranked queue manually. For each item, I will record the action, the reason code, a confidence note, and what could make the recommendation wrong.

The review is intended to identify weak or misleading picks in the rule rather than prove that every recommendation is correct.



In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-07 Section 3
# Generate the top-20 review table.

top20 = queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["score"] >= 4,
    "Higher rule-based priority because visibility and weak position are both present.",
    "Lower confidence because fewer rule conditions are satisfied."
)

top20["what_would_make_it_wrong"] = (
    "High impressions may not mean improvement is actionable; "
    "position can also be affected by query mix and other factors."
)

review_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "score",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,rank,client_hash_id,content_hash_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,client_23a62021009f63c4,content_36e53e9c707674fc,PRIORITIZE_REVIEW,visible_but_weak_position,5,Higher rule-based priority because visibility ...,High impressions may not mean improvement is a...
1,2,client_20259bd6705d81d4,content_82e35c4845e6c391,PRIORITIZE_REVIEW,visible_but_weak_position,5,Higher rule-based priority because visibility ...,High impressions may not mean improvement is a...
2,3,client_23a62021009f63c4,content_3df3f32f3fd58dea,PRIORITIZE_REVIEW,visible_but_weak_position,5,Higher rule-based priority because visibility ...,High impressions may not mean improvement is a...
3,4,client_23a62021009f63c4,content_df47d1b976106de4,PRIORITIZE_REVIEW,visible_but_weak_position,5,Higher rule-based priority because visibility ...,High impressions may not mean improvement is a...
4,5,client_23a62021009f63c4,content_bdf60c86117079be,PRIORITIZE_REVIEW,visible_but_weak_position,5,Higher rule-based priority because visibility ...,High impressions may not mean improvement is a...
5,6,client_23a62021009f63c4,content_661a7734f691bef5,PRIORITIZE_REVIEW,visible_but_weak_position,5,Higher rule-based priority because visibility ...,High impressions may not mean improvement is a...
6,7,client_23a62021009f63c4,content_cae701a83cad5e36,PRIORITIZE_REVIEW,visible_but_weak_position,5,Higher rule-based priority because visibility ...,High impressions may not mean improvement is a...
7,8,client_23a62021009f63c4,content_559cdd76da9306de,PRIORITIZE_REVIEW,visible_but_weak_position,5,Higher rule-based priority because visibility ...,High impressions may not mean improvement is a...
8,9,client_20259bd6705d81d4,content_9fff53e827550f9d,PRIORITIZE_REVIEW,visible_but_weak_position,5,Higher rule-based priority because visibility ...,High impressions may not mean improvement is a...
9,10,client_fef1a8f436438636,content_ba462518dad435fc,PRIORITIZE_REVIEW,visible_but_weak_position,5,Higher rule-based priority because visibility ...,High impressions may not mean improvement is a...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


### Weak picks

The weakest recommendations are likely to be cases where high impressions or a weak average position occur for reasons that the simple rule cannot observe. For example, high impressions do not guarantee that the content is strategically important, and average position can vary with query mix and search demand.

These cases show that the baseline is a prioritization heuristic rather than a complete decision system.

### Leakage check

I deliberately exclude label-derived fields and future-window information from the rule. In particular, `trend_pct`, `trend_direction`, and `is_declining_label` are not used. The rule uses only March 2026 performance measurements available in the development window.

No product-decision flags or client-identifying information are used as predictive features.



In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-07 Section 4
# Inspect weak picks and verify that known leakage fields
# are not present in the queue.

print("Lowest-scoring examples:")
display(
    queue.sort_values(
        ["score", "impressions"],
        ascending=[True, False]
    ).head(10)
)


# Known label-derived fields from the FlyRank data contract.
known_leakage_fields = {
    "trend_pct",
    "trend_direction",
    "is_declining_label"
}

queue_columns = set(queue.columns)

leakage_present = sorted(
    known_leakage_fields.intersection(queue_columns)
)

print("\nKnown label-derived fields in final queue:")
print(leakage_present)

if leakage_present:
    raise AssertionError(
        f"Leakage detected in final queue: {leakage_present}"
    )

print("PASS: No known label-derived fields are present in the final queue.")


# Check that the final queue contains only the intended fields.
print("\nFinal queue columns:")
print(queue.columns.tolist())

Lowest-scoring examples:


,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions,clicks,avg_position
112904,112905,client_400c21c81c8b46ef,content_21877df3f2a41464,0,review_other,MONITOR,499.0,0.0,5.903897
112905,112906,client_23a62021009f63c4,content_ed3753db29d01095,0,review_other,MONITOR,499.0,0.0,13.462588
112906,112907,client_73cda7b4e4f265ea,content_45509246babfdbf2,0,review_other,MONITOR,499.0,0.0,19.299840
112907,112908,client_ff644d8251367cbb,content_4cb537c126226e19,0,review_other,MONITOR,499.0,0.0,5.983887
112908,112909,client_ff644d8251367cbb,content_cd187f21d26decc5,0,review_other,MONITOR,499.0,0.0,15.332245
112909,112910,client_3f0ce4d44fe94f3d,content_11647416159eec65,0,review_other,MONITOR,499.0,0.0,4.684617
112910,112911,client_fef1a8f436438636,content_de636e225a2ee508,0,review_other,MONITOR,499.0,0.0,8.780975
112911,112912,client_fef1a8f436438636,content_c3fbaffc1f8ae968,0,review_other,MONITOR,499.0,0.0,15.403453
112912,112913,client_62f4a7e64f5e0096,content_4ff2446fcb03351d,0,review_other,MONITOR,499.0,0.0,11.231353
112913,112914,client_fef1a8f436438636,content_63b6b8375c6ff862,0,review_other,MONITOR,499.0,0.0,7.819556



Known label-derived fields in final queue:
[]
PASS: No known label-derived fields are present in the final queue.

Final queue columns:
['rank', 'client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action', 'impressions', 'clicks', 'avg_position']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.